# Phase 3 – Acquisition des données

L'acquisition des données constitue le point d'entrée du cycle de vie des données du projet.

Son objectif est de collecter, stocker et tracer les données nécessaires aux analyses d'investissement, à la construction de portefeuille et à la mesure des risques.

Cette phase vise à :

- identifier les sources pertinentes ;
- acquérir les données nécessaires ;
- automatiser les extractions ;
- assurer la traçabilité des opérations ;
- alimenter les phases suivantes du projet.

---

# 3.1 Sources de données

Les sources utilisées pour l'acquisition des datasets sont référencées dans le Source Registry.

| Dataset | Source principale |
|----------|----------|
| Benchmark | iShares SDG ETF |
| Securities | OpenFIGI |
| Market | Yahoo Finance |
| Fundamentals | Rapports annuels |
| Macro | FRED |
| FX | Swiss National Bank |
| ODD | Sources publiques et rapports d'entreprise |
| Exclusion | Sources ESG publiques |

Le détail des sources, fournisseurs, méthodes d'acquisition et fréquences de mise à jour est documenté dans le Source Registry.

---
# 3.2 Datasets à acquérir

| Dataset | Contenu principal | Stockage |
|----------|----------|----------|
| Benchmark Holdings | Composition du benchmark | `data/01_raw/benchmark/` |
| Securities Candidates | Référentiel des instruments financiers | `data/01_raw/securities/` |
| Market Dataset | Données de marché historiques | `data/01_raw/market/` |
| Fundamentals Dataset | Données financières des entreprises | `data/01_raw/fundamentals/` |
| Macro Dataset | Indicateurs macroéconomiques | `data/01_raw/macro/` |
| FX Dataset | Taux de change historiques | `data/01_raw/fx/` |
| ODD Dataset | Données relatives aux ODD | `data/01_raw/odd/` |
| Exclusion Dataset | Exclusions et controverses | `data/01_raw/exclusions/` |

---

# 3.3 Pipeline d'acquisition

```text
Sources externes
        │
        ▼

Acquisition
(src/acquisition/)
        │
        ▼

Couche RAW
(data/01_raw/)
        │
        ▼

Préparation
(src/preparation/)
        │
        ▼

SQLite Database
(database/sdg_investment.db)
        │
        ▼

Phase 4 – Data Quality
```

L'orchestrateur est `python -m src.pipeline.refresh`. Il lit la colonne `Frequency` du registre pour chaque jeu actif. Un jeu dû est téléchargé ou reconstruit, et son fichier brut est remplacé. Un jeu qui n'est pas dû garde le fichier déjà présent. Le calendrier est celui du journal d'acquisition, en UTC.

| Fréquence | Le jeu est dû |
|---|---|
| Daily | aucune acquisition réussie aujourd'hui |
| Monthly | aucune ce mois-ci |
| Periodic | aucune ce mois-ci. Le registre ne donne pas de jour précis |
| Quarterly | aucune ce trimestre |
| Annual | aucune cette année |
| On demand | le fichier n'existe pas encore, ou une entrée directe est acquise pendant cette exécution |

L'ordre suit `Input Dataset IDs`. Après les acquisitions :

1. chargement SQLite des jeux marqués Load Enabled ;
2. devise de cotation Yahoo pour un symbole du master qui n'est pas encore enregistré ;
3. cours CHF, univers CHF et SARON traité ;
4. trois portefeuilles ;
5. contrôle qualité.

`--force` ignore la fréquence et acquiert tous les jeux actifs. Un échec sur un jeu n'empêche pas les autres, puis la couche publiée est reconstruite avec les fichiers présents. La commande se termine alors avec un code d'erreur.

Les cours de marché viennent de `DS-004`, Yahoo Finance.

Les traitements d'acquisition alimentent simultanément le Technical Log et l'Acquisition Log afin d'assurer la traçabilité technique et métier des opérations réalisées.  

---
# 3.4 Livrables

La phase d'acquisition produit les livrables suivants :

- outils de gouvernance : Source Registry et Acquisition Log ;
- scripts d'acquisition, de préparation et de chargement des données ;
- outils d'exploitation : Technical Log ;
- datasets RAW ;
- couche de stockage SQLite ;
- contrôles et validation des données acquises ;
- documentation technique.
---

# 3.5 Documents associés

## Source Registry

Référentiel centralisant l'ensemble des sources de données utilisées dans le projet.

```text
audit/source_registry.xlsx
```

---

## Technical Log

Journal de traçabilité technique des traitements exécutés par la plateforme.

```text
logs/technical.log
```

---

## Acquisition Log

Journal de traçabilité métier des acquisitions réalisées.

```text
audit/acquisition_log.xlsx
```

---

## SQLite Database

Base de données centrale utilisée pour stocker les datasets acquis et alimenter les phases suivantes du projet.

```text
database/sdg_investment.db
```

Tables initiales :

```text
security_candidates
securities_master
securities_unmatched
```

---

## Validation Workbook

Les contrôles réalisés sur les acquisitions et les datasets produits sont documentés dans :

```text
tests/acquisition/phase_03_validation.ipynb
```

---

## Technical Documentation

Les choix d'architecture, les composants développés et les implémentations techniques sont documentés dans :

```text
03_technical_documentation.ipynb
```

# Conclusion

Cette phase établit les fondations de la plateforme de données du projet.

Les acquisitions réalisées ont permis la constitution du Benchmark Dataset, du Security Master et de la base de données SQLite utilisée comme source de référence pour les phases suivantes.

Les prochaines phases s'appuieront sur cette architecture afin de mettre en place le Data Quality Framework avant l'intégration progressive de nouvelles sources de données.